## Text input

https://platform.openai.com/docs/models

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from langchain.agents import create_agent


import os 
from langchain_openai import AzureChatOpenAI

model = AzureChatOpenAI(
    azure_deployment=os.getenv("OPENAI_DEFAULT_DEPLOYMENT"),  # or your deployment
    api_version="2025-04-01-preview",  # or your api version
    azure_endpoint=os.getenv("OPENAI_ENDPOINT"),  # or your endpoint
)

agent = create_agent(
    model=model,
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [5]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

Name: Selene Prime (colloquially “Sel’Pri”)

Location: Rim of Shackleton crater on the Moon’s south polar plateau, built on a band of peaks that catch near-continuous sunlight and look down into permanently shadowed cold-traps rich in water ice. The site gives Selene Prime steady solar power, easy access to volatiles, and an unobstructed line-of-sight to Earth and the Earth–Moon Lagrange relay stations.

Why it’s the capital: Selene Prime was chosen for a balance of resources, communications, and symbolism. Where the first permanent habitats met the shadowed ice, governments, corporations and scientific academies agreed on a neutral, highly defensible site that could sustain a city without constant resupply from Earth. It became the seat of the Selenic Assembly—the governing council that represents the Moon’s settlements.

Layout and architecture
- Surface neighborhoods are protected behind layered regolith berms and translucent ceramic domes that filter micrometeorites and solar wind.

## Image input

In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

In [ ]:
print(uploader.value)

In [ ]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [ ]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

## Audio input

In [ ]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

In [ ]:
agent = create_agent(
    model='gpt-4o-audio-preview',
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)